# Foundations

## Round 1 Targeted exercises

In [0]:
from pyspark.sql import Row

# Setup for SQL and PySpark foundation exercises
history_data = [
    Row(user_id="u001", track_id="t100", play_duration_sec=180),
    Row(user_id="u002", track_id="t101", play_duration_sec=210),
    Row(user_id="u001", track_id="t101", play_duration_sec=45),
    Row(user_id="u003", track_id="t102", play_duration_sec=300),
    Row(user_id="u002", track_id="t100", play_duration_sec=190),
    Row(user_id="u004", track_id="t100", play_duration_sec=None)
]
tracks_data = [
    Row(track_id="t100", artist_name="The Pythons", genre="Rock"),
    Row(track_id="t101", artist_name="SQL Syndicate", genre="Electronic"),
    Row(track_id="t102", artist_name="Data Bricks", genre="Rock")
]

df_history = spark.createDataFrame(history_data)
df_tracks = spark.createDataFrame(tracks_data)

df_history.createOrReplaceTempView("listening_history")
df_tracks.createOrReplaceTempView("tracks")

###Phase 1: Python Essentials
We are focusing on raw Python structures and API requests. Do this in a standard Python cell.

Exercise 1.1: The API Call. Use the requests library to make a GET request to this public placeholder API endpoint: https://jsonplaceholder.typicode.com/users. Parse the JSON response and assign it to a variable called raw_api_data.



In [0]:
import requests, json
raw_api_data = requests.get('https://jsonplaceholder.typicode.com/users').json()

print(f"Sucess! We have a {type(raw_api_data)} of data")

"""Printing list extyracted for the JSON not easy to see structure
for listValue in raw_api_data:
    print(listValue)
""" 

# Using json.dumps() to make the resulting JSON string human-readable by adding line breaks and four spaces of indentation per nesting level.
formatted_data = json.dumps(raw_api_data, indent=4)
print(formatted_data)

Exercise 1.2: Dictionaries & Lists. Write a function named extract_user_cities(data) that takes your raw_api_data (which is a list of dictionaries) as an argument. The function must iterate through the data and return a standard Python list containing only the names of the cities those users live in.

In [0]:
def extract_user_cities(data):
    user_cities = []
    for user in data:
        user_cities.append(user["address"]["city"])
    return user_cities

user_cities = extract_user_cities(raw_api_data)
print(user_cities)

### Phase 2: The SQL Bedrock
Your coach said no advanced stuff, just mastery of the core. Use %sql cells in Databricks for these. You will query the listening_history and tracks views you created in the setup cell.

Exercise 2.1: The Core Join & Aggregate. Write a SQL query that returns the artist_name and the total combined play_duration_sec for all of their tracks. You will need to join the two tables and group the results.

In [0]:
%sql
with cte(
  select track_id, sum(play_duration_sec) as total_play_time 
  from listening_history group by track_id
) select t.artist_name, c.total_play_time from tracks t left join cte c on t.track_id = c.track_id


In [0]:
%sql
select * from listening_history limit 10

In [0]:
%sql
select * from tracks limit 10;

In [0]:
%sql
SELECT 
    t.artist_name, 
    SUM(l.play_duration_sec) AS total_play_time 
FROM tracks t 
JOIN listening_history l 
    ON t.track_id = l.track_id
GROUP BY t.artist_name;

Exercise 2.2: Filtering and Shaping. Write a SQL query to find out how many unique users listened to the "Rock" genre.

In [0]:
%sql
select count(distinct(user_id)) as unique_user_rock 
from listening_history l join tracks t on l.track_id = t.track_id 
where t.genre = 'Rock'

### Phase 3: PySpark Fundamentals
PySpark is built for scale. Time to use the DataFrame API using the df_history and df_tracks dataframes generated in the setup cell. Use standard Python cells for this.

Exercise 3.1: Safe Fetching. Take df_history and create a new dataframe called clean_history. Filter out any rows where play_duration_sec is null (because bad data blows up pipelines).



In [0]:
import pyspark.sql.functions as F
df_clean_history = df_history.filter(F.col("play_duration_sec").isNotNull())
df_clean_history.display()
#display(df_clean_history)

Exercise 3.2: PySpark Aggregation. Using your new clean_history dataframe, group the data by user_id, calculate the average play_duration_sec per user, and rename that new aggregated column to avg_listen_time. Finally, display the results using Databricks' built-in display() function.

In [0]:
df_clean_history.groupBy(F.col("user_id"))\
    .agg(F.avg("play_duration_sec").alias("avg_play_duration"))\
        .display()

## Round 2: Targeted exercises

### Phase 1: Python Essentials (API & Data Parsing)
You've proven you can extract simple data. Let's add a layer of logic. We are going to hit a different endpoint on the same test API.

Exercise 1.3: Targeted Extraction. Make a GET request to https://jsonplaceholder.typicode.com/posts and parse the JSON.

In [0]:
raw_api_data = requests.get('https://jsonplaceholder.typicode.com/posts').json()

print(f"Sucess! We have a {type(raw_api_data)} of data")

# Using json.dumps() to make the resulting JSON string human-readable by adding line breaks and four spaces of indentation per nesting level.
formatted_data = json.dumps(raw_api_data, indent=4)
print(formatted_data)

Exercise 1.4: Filtering a List of Dictionaries. Write a function called get_user_posts(data, target_user_id) that takes your raw API data and an integer target_user_id. The function should iterate through the data and return a list containing only the title strings of the posts written by that specific userId. (Test it by passing in userId 3).

In [0]:

def get_user_posts(data, target_user_id):
    titles = []
    for post in data:
        if post["userId"] == target_user_id:
            titles.append(post["title"])
    return titles

user_posts = get_user_posts(raw_api_data, 3)
print(user_posts)
"""
formatted_user_posts = json.dumps(user_posts, indent=4)
print(formatted_user_posts)
"""


### Phase 2: The SQL Bedrock (Joins & Ordering)
Let's get back into the %sql cells using your listening_history and tracks tables. Time to find out what your listeners actually like.

Exercise 2.3: The Popularity Query. Write a SQL query to find out which music genre is the most popular based on the total number of times tracks in that genre were played (we are counting individual plays here, not the duration).

In [0]:
%sql
--Assuming null values in play_duration_sec doesnt count  
SELECT t.genre, COUNT(*) AS total_plays 
FROM listening_history l JOIN tracks t ON l.track_id = t.track_id 
GROUP BY t.genre 
ORDER BY total_plays DESC LIMIT 1

Exercise 2.4: Ordering. Modify that same query to order the results from the highest number of plays down to the lowest.

In [0]:
%sql
--Assuming null values in play_duration_sec doesnt count  
SELECT t.genre, COUNT(*) AS total_plays 
FROM listening_history l JOIN tracks t ON l.track_id = t.track_id 
GROUP BY t.genre 
ORDER BY total_plays DESC

### Phase 3: PySpark Fundamentals (Dataframe Joins)
You know how to aggregate a single PySpark dataframe. Now you need to combine them, just like you did in SQL, but using the PySpark DataFrame API. Continue using standard Python cells with df_history and df_tracks.

Exercise 3.3: The PySpark Join. Create a new dataframe called df_enriched. Join df_history and df_tracks together based on the track_id column. (Make sure to do an inner join, which is the default).

In [0]:
#One way to do it
#df_enriched = df_history.join(df_tracks, df_history.track_id == df_tracks.track_id, "inner")

#Another way ot do it
#df_enriched = df_history.alias("h").join(df_tracks.alias("t"),F.col("h.track_id") == F.col("t.track_id"), "inner")

# The cleanest standard for identical column names
df_enriched = df_history.join(df_tracks, "track_id", "inner")

df_enriched.display()

Assuming df_tracks has a column named 't_id' instead of 'track_id'

df_enriched = df_history.join(
    df_tracks, 
    df_history.track_id == df_tracks.t_id, 
    "inner"
).drop(df_tracks.t_id)

Notice the .drop(df_tracks.t_id) at the end. When you join on columns with different names using ==, PySpark keeps both columns in the new dataframe. A professional pipeline always drops the redundant right-side key immediately so it doesn't clutter downstream processes or cause confusion later.

Exercise 3.4: Aggregation on Joined Data. Using your new df_enriched dataframe, group by genre and calculate the average play_duration_sec for each genre. Display the final result.

In [0]:
df_enriched.groupBy(F.col("genre")).agg(F.avg("play_duration_sec").alias("avg_plays_duration")).display()

## Round 3: Targeted exercises

### Phase 1: Python Essentials (Messy Strings)
APIs rarely give you data perfectly formatted for your database. You need to clean it in transit.

Exercise 1.5: String Cleaning. Make a GET request to https://jsonplaceholder.typicode.com/users and parse the JSON. Write a function called clean_company_names(data). This function must iterate through the users, extract the company -> name string, convert the entire string to UPPERCASE, and replace any hyphens (-) with spaces. Return a list of these cleaned company names.

In [0]:
# 1. Fetch the data
raw_api_data = requests.get('https://jsonplaceholder.typicode.com/users').json()

# 2. Define the cleaning function
#Exercise solution
def clean_company_names_one_way(data):
    cleaned_names = []
    for user in data:
        # Extract the nested company name
        raw_name = user["company"]["name"]
        
        # Chain the string methods: uppercase it, then replace hyphens
        clean_name = raw_name.upper().replace("-", " ")
        
        cleaned_names.append(clean_name)
        
    return cleaned_names

#More realistic type of function
def clean_company_names(data):
    cleaned_names = []
    for user in data:
        # Safely get the data, defaulting to an empty string if it's missing
        company_info = user.get("company", {})
        raw_name = company_info.get("name")
        
        # Only apply string methods if raw_name actually exists
        if raw_name:
            clean_name = raw_name.upper().replace("-", " ")
            cleaned_names.append(clean_name)
        else:
            # Handle the bad data (e.g., log it, append "UNKNOWN")
            cleaned_names.append("UNKNOWN")
            
    return cleaned_names

# 3. Execute and test
final_company_names = clean_company_names(raw_api_data)
print(final_company_names)

### Phase 2: The SQL Bedrock (Conditional Logic)
You know how to group and aggregate. Now you need to create your own categories on the fly using CASE WHEN, which is the absolute workhorse of SQL data shaping.

Exercise 2.5: The CASE WHEN. Write a SQL query against the listening_history view. Select the track_id and play_duration_sec. Create a third, newly derived column called track_length_category.

If play_duration_sec is less than 120, label it 'Short'.

If it is between 120 and 240 (inclusive), label it 'Medium'.

If it is greater than 240, label it 'Long'.

Hint: Think about how you want your CASE WHEN to handle those pesky null values so they don't accidentally get labeled!

In [0]:
%sql
SELECT track_id, play_duration_sec, 
CASE 
WHEN play_duration_sec > 240 THEN 'Long' 
WHEN play_duration_sec BETWEEN 120 AND 240 THEN 'Medium'
WHEN play_duration_sec < 120 THEN 'Short'
ELSE 'Unkown' END AS track_length_category
FROM listening_history
    

### Phase 3: PySpark Fundamentals (Mutations & Missing Data)
Sometimes dropping nulls (like you did with .filter(F.col(...).isNotNull())) destroys valuable data. If a track duration is null, maybe we just want to assume it was skipped at 0 seconds.

Exercise 3.5: Filling Nulls. Create a new dataframe called df_filled from your original df_history. Use the PySpark .fillna() method to replace any null values in the play_duration_sec column with 0.

In [0]:
df_filled = df_history.fillna({'play_duration_sec': 0})

Example with multiple columns

df_filled = df_history.fillna({'play_duration_sec': 0,
                               'other_columns': False,
                               'another_column': 'uknown'})

Exercise 3.6: Column Math. Data scientists usually want minutes, not seconds. Using your new df_filled dataframe, use the .withColumn() method to create a brand new column called play_duration_min. Calculate this by dividing play_duration_sec by 60. Display the final dataframe.

In [0]:
df_filled.withColumn('play_duration_min', F.col("play_duration_sec") / 60).display()